# Imports

In [1]:
import cda2
import datetime
import pyspark.sql.functions as F
import pyspark.sql.types as T
import json
import re

from pyspark.sql.window import Window
from pyspark.sql.functions import col, row_number

# Connect to Spark

In [2]:
api = cda2.Api()

Set configuration parameters to better optimize queries.

In [3]:
config = {
    "spark.sql.adaptive.enabled": "true",
    "spark.sql.adaptive.coalescePartitions.enabled": "true",
    "spark.sql.adaptive.coalescePartitions.parallelismFirst": "false",
    "spark.sql.adaptive.coalescePartitions.minPartitionSize": "1m",
    "spark.executor.memory": "8g",
    "spark.executor.memoryOverhead": "16g",
}

Start Spark and specify number of cpus to use. 400 is quite high, but we'll be running 1 year at a time and want to have it done in just a few minutes.

In [4]:
api.start_spark(n_executors=400, config=config)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


23/11/14 18:14:09 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
23/11/14 18:14:11 WARN DomainSocketFactory: The short-circuit local reads feature cannot be used because libhadoop cannot be loaded.
23/11/14 18:14:29 WARN YarnSchedulerBackend$YarnSchedulerEndpoint: Attempted to request executors before the AM has registered!


# Set Parameters

Set date range and airports here.

In [5]:
year0 = "2019"
year1 = str(int(year0) + 1)

In [6]:
datelist = [
    year0 + "1101",
    year0 + "1201",
]

datelist = [
    year0 + "0101",
    year0 + "0201",
    year0 + "0301",
    year0 + "0401",
    year0 + "0501",
    year0 + "0601",
    year0 + "0701",
    year0 + "0801",
    year0 + "0901",
    year0 + "1001",
    year0 + "1101",
    year0 + "1201",
    year1 + "0101",
]

In [7]:
airports = [
    "KADW",
    "KATL",
    "KBOS",
    "KBWI",
    "KCLT",
    "KDCA",
    "KDEN",
    "KDFW",
    "KDTW",
    "KEWR",
    "KFLL",
    "KIAD",
    "KIAH",
    "KJFK",
    "KLAS",
    "KLAX",
    "KLGA",
    "KMCO",
    "KMDW",
    "KMEM",
    "KMIA",
    "KMSP",
    "KORD",
    "KPHL",
    "KPHX",
    "KSAN",
    "KSDF",
    "KSEA",
    "KSFO",
    "KSLC",
    "KTPA",
    "PANC",
    "PHNL",
]

In [8]:
# airports = [ "KMEM", ]

# define functions

Function to convert Unix timestamp (milliseconds from 1970) to YYYYMMDD string.

In [9]:
@F.udf("string")
def to_date(ts):
    return datetime.datetime.utcfromtimestamp(ts / 1000).strftime("%Y%m%d")

In [10]:
aa_point_json =  '{"fields":[{"metadata":{},"name":"entry_point","nullable":true,"type":{"fields":[{"metadata":{},"name":"primary_key","nullable":true,"type":"string"},{"metadata":{},"name":"time","nullable":true,"type":"long"},{"metadata":{},"name":"latitude","nullable":true,"type":"double"},{"metadata":{},"name":"longitude","nullable":true,"type":"double"},{"metadata":{},"name":"altitude","nullable":true,"type":"float"},{"metadata":{},"name":"course","nullable":true,"type":"float"},{"metadata":{},"name":"speed","nullable":true,"type":"float"},{"metadata":{},"name":"source_point_keys","nullable":true,"type":{"containsNull":true,"elementType":"string","type":"array"}},{"metadata":{},"name":"airspace_key","nullable":true,"type":"string"},{"metadata":{},"name":"distance","nullable":true,"type":"double"},{"metadata":{},"name":"climb_rate","nullable":true,"type":"double"}],"type":"struct"}}],"type":"struct"}'

In [11]:
aa_point_schema = T.StructType.fromJson(json.loads(aa_point_json))

functions to stringify the airspace assignment row and the entry/exit point columns

In [12]:
def strinigfy_aa_point(aa_point:aa_point_schema) -> str:
    output = ""
    output += str(aa_point.time) + " "
    output += f'{aa_point.latitude:.6f}' + " "
    output += f'{aa_point.longitude:.6f}' + " "
    output += f'{aa_point.altitude:.0f}'
    return output

strinigfy_aa_point_udf = F.udf(strinigfy_aa_point, T.StringType())

In [13]:
none_string = "*"

def strinigfy_aa(artcc:T.StringType, airspace_identifier:T.StringType, airspace_class:T.StringType, airspace_subclass:T.StringType, entry_point:aa_point_schema, exit_point:aa_point_schema, duration:T.DoubleType, distance:T.DoubleType) -> str:
    output = ""
    output += artcc + " "
    output += "'" + airspace_identifier + "' "
##    output += airspace_class + " "
#    output += airspace_subclass + " "
    output += (none_string if airspace_subclass is None else airspace_subclass) + " "
    output += strinigfy_aa_point(entry_point) + " "
    output += strinigfy_aa_point(exit_point) + " "
    output += f'{duration:.1f}' + " "
    output += f'{distance:.1f}'
    return output

strinigfy_aa_udf = F.udf(strinigfy_aa, T.StringType())

# main loop to do one month at a time

In [14]:
def retrieve_a_month(i:T.IntegerType):
    
    dates = {"start_date": datelist[i], "end_date": datelist[i+1]}
    print(dates)

    
    ## get the raw AA for the date range
    df_aa_raw = (
        api.dataframe("AirspaceAssignment", **dates, partition_filters=[
            api.custom_partitions(["ASSOCIATED", "StaticEramAirspace", "SECTOR"]),
            api.custom_partitions(["ASSOCIATED", "StaticEramAirspace", "TRACON"]),
            api.custom_partitions(["ASSOCIATED", "FlightInformationRegion"]),
        ], metadata=True)
    )


    ## define the point schema
    if i == 0:
        df_aa_point = (df_aa_raw.select("entry_point"))
        aa_point_json = df_aa_point.schema.json()   
        aa_point_schema = T.StructType.fromJson(json.loads(aa_point_json))


    ## sort the df_aa_raw data by track_key and entry_point_time,
    ## create an "artcc" column from the meta_custom_partitions[1],
    ## and stringify the needed data to an "airspace_assignment" column
    df_aa = (
        df_aa_raw
        .sort(F.col("track_key"), F.col("entry_point.time"))
        .withColumn("artcc", df_aa_raw.metadata.custom_partitions[1])
        .withColumn("airspace_assignment", strinigfy_aa_udf("artcc", "airspace_identifier", "airspace_class", "airspace_subclass", "entry_point", "exit_point", "known_visit_duration", "known_distance_covered"))
        .select(
            "track_key",
            "airspace_assignment",
        )
    )     


    ## group by track_key and concatenate grouped "airspace_assignment"
    df_aa_grouped = (
        df_aa.groupBy("track_key").agg(F.concat_ws(":", F.collect_list("airspace_assignment")).alias("airspace_assignments"))
    )


    ## get flightplans for the arrivals to and departures from airports_of_interest
    df_fps = (
        api.dataframe("FlightplanSeries", **dates, partition_filters=api.custom_partitions("ASSOCIATED"), metadata=True)
        .select(
            "track_key",
            to_date("metadata.effective_start_date").alias("start_date"),
            to_date("metadata.effective_end_date").alias("end_date"),
            "callsign",
            "aircraft_type",
            "mode_s_code",
            F.col("initial_departure_aerodrome").alias("orig"),
            F.col("final_destination_aerodrome").alias("dest"),
        )
        .withColumn("month", F.substring("start_date", 5, 2))
        .filter(F.col("orig") != F.col("dest"))
        .filter(F.col("orig").isin(airports) | F.col("dest").isin(airports))
    )


    ## create the arrivals and departures, then combine them
    df_arrivals = (
        df_fps
            .withColumn("airport", F.col("dest"))
            .withColumn("operation", F.lit("ARRIVALS"))
            .filter(F.col("airport").isin(airports))
    )

    df_departures = (
        df_fps
            .withColumn("airport", F.col("orig"))
            .withColumn("operation", F.lit("DEPARTURES"))
            .filter(F.col("airport").isin(airports))
    )

    df_fps_combined = df_arrivals.unionByName(df_departures)


    ## join AA with FPS to filter aa based on the track_keys in the flightplans for the cities we care about
    df_aa_joined = (
        df_aa_grouped
        .join(df_fps_combined, on=["track_key"], how="inner")
    )

    ## output the result
    df_aa_output = (
        df_aa_joined
        .select(
            "track_key",
            "start_date",
            "end_date",
            "orig",
            "dest",
            "callsign",
            "mode_s_code",
            "aircraft_type",
            "airspace_assignments",
            "airport",
            "month",
            "operation",
        )
    )

    (
        df_aa_output
        .repartition("airport", "operation", "month")
        .write.option("header", True).partitionBy(["airport", "operation", "month"])
#        .csv("CRAFT/" + dates["start_date"] + "/airspace_assignments", compression="gzip", mode="append")
        .csv("CRAFT/" + year0 + "/airspace_assignments", compression="gzip", mode="append")
    )

    return

In [15]:
i = 0
while i < (len(datelist) - 1):
    retrieve_a_month(i)
    i += 1

{'start_date': '20190101', 'end_date': '20190201'}


23/11/14 18:15:10 WARN YarnScheduler: Initial job has not accepted any resources; check your cluster UI to ensure that workers are registered and have sufficient resources
23/11/14 18:15:25 WARN YarnScheduler: Initial job has not accepted any resources; check your cluster UI to ensure that workers are registered and have sufficient resources
23/11/14 18:15:40 WARN YarnScheduler: Initial job has not accepted any resources; check your cluster UI to ensure that workers are registered and have sufficient resources


{'start_date': '20190201', 'end_date': '20190301'}


{'start_date': '20190301', 'end_date': '20190401'}


Multiple versions found: 3.1.15.1, 3.1.34
Multiple versions found: 3.1.20, 3.1.34                                         


...

Multiple versions found: 3.1.20, 3.1.61                                         


{'start_date': '20190901', 'end_date': '20191001'}


{'start_date': '20191001', 'end_date': '20191101'}


{'start_date': '20191101', 'end_date': '20191201'}


{'start_date': '20191201', 'end_date': '20200101'}
